<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2012/Decision%20Boundaries_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Required Assignment 12.2: Decision Boundaries

**Estimated Time: 60 Minutes**

**Total Points: 55**

This activity focuses on the effect of changing your decision threshold and the resulting predictions.  Again, you will use the `KNeighborsClassifier`, but this time you will explore the `predict_proba` method of the fit estimator to change the thresholds for classifying observations.  You will explore the results of changing the decision threshold on the false negative rate of the classifier for the insurance data.  Here, we suppose the important thing is to not make the mistake of predicting somebody would not default when they really do.  

#### Index

- [Problem 1](#Problem-1)
- [Problem 2](#Problem-2)
- [Problem 3](#Problem-3)
- [Problem 4](#Problem-4)
- [Problem 5](#Problem-5)
- [Problem 6](#Problem-6)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn import set_config
import warnings
warnings.filterwarnings("ignore")
set_config(display="diagram")



### The Dataset

You continue to use the default example, and the data is again loaded and split for you below.

In [4]:
default = pd.read_csv('/content/sample_data/default.csv')

In [5]:
default.head(2)

,Unnamed: 0,default,student,balance,income
0,1,No,No,729.526495,44361.625074
1,2,No,Yes,817.180407,12106.134700


In [6]:
X_train, X_test, y_train, y_test = train_test_split(default.drop(columns ={'default'}),  default['default'], random_state=42)

In [7]:
transformer= make_column_transformer((OneHotEncoder(drop='if_binary'),['student']), remainder = StandardScaler())


[Back to top](#-Index)

### Problem 1

#### Basic Pipeline

**10 Points**

Use the `Pipeline` function to create a pipeline `base_pipe` with steps `transformer` and `knn`. Assign `transformer` to `'transformer'` and assign a `KNeighborsClassifier()` with `n_neighbors = 10` to `'knn'`.

In [8]:
### GRADED
base_pipe = ''
# YOUR CODE HERE
base_pipe= Pipeline(steps =
                    [
                        ("transformer", transformer) , ("knn",KNeighborsClassifier(n_neighbors=10))
                    ])
# Answer check
base_pipe

Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder=StandardScaler(),
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['student'])])),
                ('knn', KNeighborsClassifier(n_neighbors=10))])

[Back to top](#-Index)

### Problem 2

#### Accuracy of KNN with 50% probability boundary

**10 Points**

- Use the `fit` function to train `base_pipe` on `X_train` and `y_train`.
- Use the `score` function to calculate the performance of `base_pipe` on the test sets. Assign the result to `base_acc`.
- Use the `predict` function on `base_pipe` to make predictions on `X_test`. Assign the reusl to `preds`.
- Initialize the `base_fn` variable to `0`.
- Use a `for` loop to loop over `zip(preds, y_test)`. Inside the `for` loop:
    - Use an `if` block to determine the accuracy for this default setting and assign it to `base_acc`. Also, consider the proportion of false negatives here.  Assign these as `base_fn`.  

In [9]:
### GRADED
base_acc = ''
base_fn = ''
# YOUR CODE HERE
base_pipe.fit(X_train,y_train)
base_acc= base_pipe.score(X_test, y_test)
preds = base_pipe.predict(X_test)
base_fn = 0
pred_acc=0
# accuracy is defined as number of preds that were correct by the model
#FN is defined as number of predictions which were predicted False (No) but were in reality positive.
for i in zip(preds, y_test):
     if ((i[0] == i[1])):
         pred_acc=pred_acc+1
     if((i[0] =='No')  & (i[1] == 'Yes')):
         base_fn=base_fn+1

base_acc=pred_acc/len(preds)
# Answer check
print(base_acc)
print(base_fn)

0.9712
65


In [10]:
print(preds[:5],y_test[:5])

['No' 'No' 'No' 'No' 'No'] 6252    No
4684    No
1731    No
4742    No
4521    No
Name: default, dtype: object


In [11]:
y_test.value_counts()

,count
default,
No,2419
Yes,81


[Back to top](#-Index)

### Problem 3

#### Prediction probabilities

**10 Points**

As demonstrated in Video 12.5, your fit estimator has a `predict_proba` method that will output a probability for each observation.  


Use the `predict_proba` function on `base_pipe` to predict the probabilities on `X_test`. Assign the predicted probabilities as an array using the test data to `base_probs` below.

In [12]:
### GRADED
base_probs = ''
# YOUR CODE HERE
base_probs = base_pipe.predict_proba(X_test)
base_probs= np.array(base_probs)
# Answer check
pd.DataFrame(base_probs[:5] , columns = ["p_no","p_yes"])

,p_no,p_yes
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0


In [13]:
base_probs

array([[1. , 0. ],
       [1. , 0. ],
       [1. , 0. ],
       ...,
       [1. , 0. ],
       [1. , 0. ],
       [0.9, 0.1]])

[Back to top](#-Index)

### Problem 4

#### A Stricter `default` estimation

**10 Points**

As discussed in the previous assignment, if you aim to minimize the number of predictions that miss default observations you may consider increasing the probability threshold to make such a classification.  Accordingly, use your probabilities from the last problem to only predict 'No' if you have a higher than 70% probability that this is the label.  Assign your new predictions as an array to `strict_preds`.  Determine the number of false negative predictions here and assign them to `strict_fn` below.  

In [23]:
#FN is defined as number of predictions which were predicted False (No) but were in reality positive.
### GRADED
strict_fn = 0
# YOUR CODE HERE
base_probs= pd.DataFrame(base_probs, columns = ["p_n","p_y"])
strict_probs_df = base_probs.copy()

strict_preds = np.array(strict_probs_df)
strict_probs_df['preds'] = np.where(strict_probs_df['p_n'] >0.7, 'No', 'Yes')
strict_preds = np.array(strict_probs_df['preds'])

#FN is defined as number of predictions which were predicted False (No) but were in reality positive.
for j in zip(strict_preds, y_test):
     if((j[0] =='No')  & (j[1] == 'Yes')):
         strict_fn = strict_fn+1

# Answer check
print(strict_fn)


44


[Back to top](#-Index)

### Problem 5

#### Minimizing False Negatives

**10 Points**

Consider a 50%, 70%, and 90% decision boundary for predicting "No".  Which of these minimizes the number of false negatives?  Assign your solution as an integer -- 50, 70, or 90 -- to `ans5` below.



In [37]:
### GRADED
ans5 = ''
# YOUR CODE HERE

# calculate FN for 90% FN
strict_probs90_df = base_probs.copy()
strict_probs90_df['preds'] = np.where(strict_probs90_df['p_n'] >0.9, 'No', 'Yes')
strict90_preds = np.array(strict_probs90_df['preds'])
strict90_fn=0

#FN is defined as number of predictions which were predicted False (No) but were in reality positive.
for j in zip(strict90_preds, y_test):
     if((j[0] =='No')  & (j[1] == 'Yes')):
         strict90_fn = strict90_fn+1

low =strict90_fn
ans5=90
if (low >strict_fn):
  low=strict_fn, ans5=70
if (low >base_fn ):
  low=base_fn, ans5=50

# Answer check
print(ans5)

90


In [32]:
strict_fn, base_fn, strict90_fn

(44, 65, 22)

[Back to top](#-Index)

### Problem 6

#### Visualizing decision boundaries

**5 Points**

For this exercise, a visualization of the decision boundary using a synthetic dataset is created and plotted below.  Which of these would you choose to minimize the number of false negatives?  Enter your choice as an integer -- 1, 20, or 50 -- to `ans6` below.

<center>
    <img src = images/dbounds.png />
</center>

In [17]:
### GRADED
ans6 = ''
# YOUR CODE HERE


# Answer check
print(ans6)